In [15]:
from bs4 import BeautifulSoup
import pandas as pd
import re

HTML_FILE = "identificadores.html"
OUTPUT_CSV = "identificadores.csv"

with open(HTML_FILE, "r", encoding="utf-8") as f:
    soup = BeautifulSoup(f, "html.parser")

dados = []

TIPOS = {
    "ACERVO-A": "assunto",
    "ACERVO-C": "classificacao",
    "ACERVO-N": "nivel_geografico",
    "ACERVO-P": "periodo",
    "ACERVO-E": "periodicidade",
    "ACERVO-V": "variavel",
}

def inferir_tipo(element):
    for parent in element.parents:
        if parent.name == "div" and "ACERVO" in parent.get("class", []):
            acervo_id = parent.get("id")
            if acervo_id in TIPOS:
                return TIPOS[acervo_id]
    return "desconhecido"


# pega iden1 até iden9
for span in soup.find_all(
    "span",
    class_=re.compile(r"^iden[1-9]$")
):
    ident_id = span.get_text(strip=True)

    pai = span.parent
    texto = pai.get_text(" ", strip=True)

    nome = re.sub(rf"\b{re.escape(ident_id)}\b", "", texto)
    nome = nome.strip(" -–")

    tipo = inferir_tipo(span)

    dados.append({
        "tipo": tipo,
        "id": ident_id,
        "nome": nome
    })

df = pd.DataFrame(dados).drop_duplicates()

df = df.sort_values(["tipo", "id"]) if "tipo" in df.columns else df.sort_values("id")

df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")

print(f"CSV gerado com {len(df)} identificadores")


CSV gerado com 13328 identificadores


In [2]:
import pandas as pd



In [14]:
import pandas as pd
import pickle

# Primeiro, vamos filtrar os dados
# (Assumindo que você já tem o DataFrame 'ident')
ident = pd.read_csv('identificadores.csv')
# Converter a coluna de ID para string (se necessário)
if ident['id'].dtype in ['int64', 'float64', 'int32', 'float32']:
    ident['id_str'] = ident['id'].astype(str)
    id_column = 'id_str'
else:
    id_column = 'id'

# Filtrar APENAS os registros onde o ID começa com uma letra
df_com_letra = ident[ident[id_column].str.match(r'^[A-Za-z]', na=False)].copy()

# Resetar índice
df_com_letra = df_com_letra.reset_index(drop=True)

# Criar lista de pares chave-valor para o kv_store.mset()
# Onde chave = ID e valor = nome (serializado em bytes)
kv_pairs = []

for idx, row in df_com_letra.iterrows():
    # Garantir que temos as colunas necessárias
    if id_column in row and 'nome' in row:
        chave = str(row[id_column])
        valor = str(row['nome'])

        # Adicionar ao formato requerido [chave, valor_em_bytes]
        kv_pairs.append([chave, valor.encode('utf-8')])


In [13]:
kv_pairs

[['N1', b'Brasil'],
 ['N10', b'Distrito'],
 ['N101',
  b'Pa\xc3\xads do Mercosul, Bol\xc3\xadvia e\n                    Chile'],
 ['N1011', b'Munic\xc3\xadpios Costeiros'],
 ['N1013', b'Munic\xc3\xadpios de Faixa de Fronteira'],
 ['N102', b'Bairro'],
 ['N103', b'Total das \xc3\xa1reas - POF'],
 ['N11', b'Subdistrito'],
 ['N110', b'Total das \xc3\xa1reas - PME'],
 ['N1100',
  b'Brasil, sem especifica\xc3\xa7\xc3\xa3o de Unidade da Federa\xc3\xa7\xc3\xa3o'],
 ['N1101', b'Ignorado'],
 ['N1102', b'Estrangeiro'],
 ['N1103', b'\n                    Total'],
 ['N1104',
  b'Unidade da Federa\xc3\xa7\xc3\xa3o,\n                    sem especifica\xc3\xa7\xc3\xa3o de Munic\xc3\xadpio'],
 ['N1105', b'\xc3\x81rea de influ\xc3\xaancia - PNSB'],
 ['N111', b'Unidade Federativa do Mercosul, Bol\xc3\xadvia e Chile'],
 ['N1124', b'Coordena\xc3\xa7\xc3\xa3o Regional da Funai'],
 ['N1125', b'Terra Ind\xc3\xadgena'],
 ['N1126', b'\n                    Unidade de Conserva\xc3\xa7\xc3\xa3o'],
 ['N1145', b'Ter

In [1]:
import requests
import pandas as pd

def fetch_agregados_df():
    url = "https://servicodados.ibge.gov.br/api/v3/agregados"

    # Faz a requisição GET
    resp = requests.get(url)
    resp.raise_for_status()  # dá erro se for 4xx/5xx

    data = resp.json()  # lista de pesquisas

    # Lista onde vamos acumular os agregados
    rows = []

    for pesquisa in data:
        pesquisa_id = pesquisa.get("id")
        pesquisa_nome = pesquisa.get("nome")

        # Cada pesquisa tem uma lista de agregados
        for ag in pesquisa.get("agregados", []):
            rows.append({
                "pesquisa_id": pesquisa_id,
                "pesquisa_nome": pesquisa_nome,
                "agregado_id": ag.get("id"),
                "agregado_nome": ag.get("nome")
            })

    # Monta o DataFrame
    df = pd.DataFrame(rows)
    return df

if __name__ == "__main__":
    df_agregados = fetch_agregados_df()
    print(df_agregados.head())
    df_agregados.to_csv("agregados_ibge.csv", index=False)
    print("Salvou tudo em agregados_ibge.csv tuim tuim!")


  pesquisa_id                 pesquisa_nome agregado_id  \
0          D5             Áreas Urbanizadas        8418   
1          CL  Cadastro Central de Empresas        1685   
2          CL  Cadastro Central de Empresas        1732   
3          CL  Cadastro Central de Empresas        1733   
4          CL  Cadastro Central de Empresas        1734   

                                       agregado_nome  
0  Áreas urbanizadas, Loteamento vazio, Área tota...  
1  Unidades locais, empresas e outras organizaçõe...  
2  Dados gerais das empresas por faixas de pessoa...  
3  Dados gerais das unidades locais por faixas de...  
4  Dados gerais das unidades locais por faixas de...  
Salvou tudo em agregados_ibge.csv tuim tuim!


In [2]:
df_agregados

,pesquisa_id,pesquisa_nome,agregado_id,agregado_nome
0,D5,Áreas Urbanizadas,8418,"Áreas urbanizadas, Loteamento vazio, Área tota..."
1,CL,Cadastro Central de Empresas,1685,"Unidades locais, empresas e outras organizaçõe..."
2,CL,Cadastro Central de Empresas,1732,Dados gerais das empresas por faixas de pessoa...
3,CL,Cadastro Central de Empresas,1733,Dados gerais das unidades locais por faixas de...
4,CL,Cadastro Central de Empresas,1734,Dados gerais das unidades locais por faixas de...
...,...,...,...,...
9201,SI,Sistema Nacional de Pesquisa de Custos e Índic...,33,"Custo de projeto m², por padrão de acabamento ..."
9202,SI,Sistema Nacional de Pesquisa de Custos e Índic...,34,"Preços medianos, por materiais e serviços (sér..."
9203,SI,Sistema Nacional de Pesquisa de Custos e Índic...,35,"Salários medianos, por categorias profissionai..."
9204,SI,Sistema Nacional de Pesquisa de Custos e Índic...,647,"Custo de projeto m², por tipo de projeto e pad..."
